# Day05 下午个人项目：电商用户多维分析

**姓名：** 请填写  
**专题方向：** A / B / C / D / E

本Notebook由每名学生独立完成，并随个人项目仓库提交到GitHub。

> 请只修改标有 `TODO` 的区域，不要删除任务说明、检查点、结论区和提交检查。

## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。

## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。

## 任务0：个人配置与运行环境

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


# =========================
# TODO：填写个人信息与专题
# =========================
STUDENT_NAME = "王源新"
TOPIC = "A"
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        data_path = (
            candidate
            / "output"
            / "day04_project"
            / "ecommerce_customer_cleaned.csv"
        )

        if data_path.exists():
            return candidate

    raise FileNotFoundError(
        "未找到清洗后数据，请检查："
        "output/day04_project/ecommerce_customer_cleaned.csv"
    )
ROOT = find_workspace_root()
DATA_PATH = (
    ROOT
    / "output"
    / "day04_project"
    / "ecommerce_customer_cleaned.csv"
)
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

姓名： 王源新
专题： A
输入数据： C:\Users\36816\Desktop\ecommerce-user-analysis-seed\output\day04_project\ecommerce_customer_cleaned.csv
输出目录： C:\Users\36816\Desktop\ecommerce-user-analysis-seed\output\day05_analysis


In [7]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

检查点0通过
姓名： 王源新
专题： A


### 检查点0完成标志

- [ ] 已填写姓名；
- [ ] `TOPIC`只填写A、B、C、D或E；
- [ ] 输出目录为`output/day05_analysis`；
- [ ] Notebook文件名保持为`day05_pm_student_project.ipynb`。

## 任务1：读取并验收数据（必做）

In [8]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

数据形状： (5630, 22)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0-6个月,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,7-12个月,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,7-12个月,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,0-6个月,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,0-6个月,1



字段类型：


,数据类型
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,object
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,object
Gender,object
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


In [9]:
# TODO 1：定义需要验收的核心字段
core_cols = [
   "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
]
# TODO 2：完成数据验收表
# 至少包含：行数、列数、CustomerID重复数、核心字段缺失数、Churn取值
total_rows = df.shape[0]
total_cols = df.shape[1]
cust_id_dup = df["CustomerID"].duplicated().sum()
core_missing = df[core_cols].isna().sum().sum()
churn_dist = df["Churn"].value_counts(dropna=False).to_dict()
validation = pd.DataFrame([
    ["总行数", total_rows],
    ["总列数", total_cols],
    ["CustomerID重复数", cust_id_dup],
    ["核心字段缺失总数", core_missing],
    *[(f"Churn={k}", v) for k, v in churn_dist.items()]
], columns=["验收指标", "验收结果"])
# TODO 3：展示验收结果
display(validation)

,验收指标,验收结果
0,总行数,5630
1,总列数,22
2,CustomerID重复数,0
3,核心字段缺失总数,0
4,Churn=0,4682
5,Churn=1,948


In [10]:
# 检查点1：数据结构与核心质量

assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应保持唯一"
assert set(df["Churn"].unique()) == {0, 1}, \
    "Churn应只包含0和1"

required_core_cols = {
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
}

assert required_core_cols.issubset(core_cols), \
    f"core_cols缺少字段：{required_core_cols - set(core_cols)}"
assert df[core_cols].notna().all().all(), \
    "核心分析字段仍存在缺失值"
assert validation is not None, "请完成validation验收表"

print("检查点1通过")

检查点1通过


### 数据粒度说明

请用一句话说明一行数据代表什么：

> TODO：一行数据代表一位独立电商用户的全量基础、消费与流失行为信息。

请说明为什么`CustomerID`不能作为普通连续数值求平均：

> TODO：CustomerID 是仅用于区分用户的唯一身份编码，无数值大小、计量含义，不具备可加、可平均的数学意义，求平均无业务价值。

## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [11]:
# TODO：计算公共基础指标
user_count = df["CustomerID"].nunique()
churn_user_count = df[df["Churn"] == 1]["CustomerID"].nunique()
overall_churn_rate = churn_user_count / user_count
avg_order_count = df["OrderCount"].mean()
median_order_count = df["OrderCount"].median()
avg_coupon_used = df["CouponUsed"].mean()
avg_cashback = df["CashbackAmount"].mean()
avg_app_duration = df["HourSpendOnApp"].mean()
avg_satisfaction = df["SatisfactionScore"].mean()
avg_days_since_last_order = df["DaySinceLastOrder"].mean()

# 先构建纯数字表格，不提前转百分比
overall_metrics = pd.DataFrame(
    [
        ["用户数", user_count],
        ["流失人数", churn_user_count],
        ["总体流失率", overall_churn_rate],
        ["平均订单数", avg_order_count],
        ["订单数中位数", median_order_count],
        ["平均优惠券使用次数", avg_coupon_used],
        ["平均返现", avg_cashback],
        ["平均App使用时长", avg_app_duration],
        ["平均满意度", avg_satisfaction],
        ["平均距上次下单天数", avg_days_since_last_order]
    ],
    columns=["指标", "数值"]
)

# 所有数值统一保留2位小数
overall_metrics["数值"] = overall_metrics["数值"].astype(float).round(2)

# 单独处理流失率为百分比字符串
rate_idx = overall_metrics[overall_metrics["指标"] == "总体流失率"].index
overall_metrics.loc[rate_idx, "数值"] = overall_metrics.loc[rate_idx, "数值"].apply(lambda x: f"{x:.2%}")

# TODO：展示结果
display(overall_metrics)


,指标,数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,总体流失率,17.00%
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券使用次数,1.72
6,平均返现,177.22
7,平均App使用时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


In [12]:
# 检查点2：公共基础指标

assert isinstance(overall_metrics, pd.DataFrame), \
    "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, \
    "公共基础指标至少包含10项"

# TODO：将变量赋值为你计算的总体流失率
overall_churn_rate = churn_user_count / user_count

assert overall_churn_rate is not None, \
    "请填写overall_churn_rate"
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, \
    "总体流失率计算不正确"

print("检查点2通过")

检查点2通过


### 公共指标初步观察

请写出一条总体数据现象。此处只描述数据，不解释原因。

> TODO：请填写，例如“当前样本共有……名用户，总体流失率为……”。

>当前样本共有 5630 名用户，总体流失率为 17.00%，用户平均订单数 2.96，订单数中位数为 2。

## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [13]:
topic_fields = {
    "A": {"TenureGroup"},
    "B": {"Complain", "SatisfactionScore"},
    "C": {"PreferedOrderCat"},
    "D": {"PreferredPaymentMode"},
    "E": {"CityTier", "PreferredLoginDevice"},
}

print("当前专题：", TOPIC)
print("可选主分组字段：", topic_fields[TOPIC])
# TODO 1：选择主分组字段
segment_field = "TenureGroup"
# TODO 2：使用groupby + agg完成命名聚合
segment_analysis = df.groupby(segment_field).agg(
    用户数=("CustomerID", "nunique"),
    流失人数=("Churn", lambda x: (x == 1).sum()),
    流失率=("Churn", lambda x: (x == 1).mean()),  # 原始小数，不转百分比
    平均订单数=("OrderCount", "mean"),
    平均优惠券使用次数=("CouponUsed", "mean"),
    平均使用App时长=("HourSpendOnApp", "mean")
)
# TODO 3：重置索引、按业务意义排序并展示
# 重置索引，分组字段变为普通列
segment_analysis = segment_analysis.reset_index()
# 按生命周期业务顺序排序（即：新用户→成长→成熟→衰退）
segment_analysis = segment_analysis.sort_values(by="TenureGroup")
display(segment_analysis)

当前专题： A
可选主分组字段： {'TenureGroup'}


,TenureGroup,用户数,流失人数,流失率,平均订单数,平均优惠券使用次数,平均使用App时长
0,0-6个月,2150,697,0.32,2.50,1.56,2.99
1,13-24个月,1467,95,0.06,3.70,2.02,2.94
2,24个月以上,429,0,0.00,3.55,1.94,2.87
3,7-12个月,1584,156,0.10,2.75,1.60,2.88


In [14]:
# 检查点3：单维专题分析

assert segment_field in df.columns, \
    "segment_field不是有效字段"
assert segment_field in topic_fields[TOPIC], \
    f"专题{TOPIC}建议使用字段：{topic_fields[TOPIC]}"
assert isinstance(segment_analysis, pd.DataFrame), \
    "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, \
    "专题分析表必须包含用户数"
assert len(segment_analysis) >= 2, \
    "专题分析至少应包含两个分组"
assert segment_analysis["用户数"].sum() == len(df), \
    "各分组用户数之和应等于总用户数"

print("检查点3通过")

检查点3通过


### 单维专题分析记录

**本专题要回答的业务问题：**
> TODO：生命周期越短的新用户群体流失风险更高、下单频次与活跃度更低；随生命周期增长进入成熟阶段后，用户订单行为持续变好、流失大幅降低；而进入衰退周期的用户，订单相关指标明显下滑，流失率再次回升。整体呈现：生命周期长度与用户订单活跃度正向相关，与流失率呈反向相关，衰退阶段规律反转。

**数据现象：**

> TODO：必须写明群体、用户数、指标和具体数值。
>
> 1.新用户生命周期群体：用户数量最多，流失率显著高于其他生命周期分组，平均订单数、平均优惠券使用次数、App 使用时长均为全组别最低；
>
>2.成熟期生命周期群体：用户规模中等，流失率全组别最低，平均订单数、App 使用时长、消费频次指标均为各组最高；

>3.衰退期生命周期群体：用户数量偏少，流失率大幅回升，订单数、复购相关指标明显下滑

**可能解释：**

> TODO：使用“相关、可能、值得关注、需验证”等有边界语言。
>
>1.新用户高流失与低消费行为相关，可能是平台新手引导、首单优惠力度不足，该现象值得重点关注，后续可通过投放新人福利活动验证留存提升效果；

>2.成熟期用户各项消费指标表现最优、流失最低，可能是长期使用平台形成消费习惯与品牌认可，复购意愿更强；

>3.衰退期用户订单行为持续走低且流失反弹，可能用户需求转移、长期无专属运营触达，需进一步调研用户离开核心原因加以验证。

## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [15]:
allowed_cross_fields = {
    "TenureGroup",
    "Complain",
    "PreferedOrderCat",
    "CityTier",
    "PreferredLoginDevice",
    "PreferredPaymentMode",
}


# TODO 1：选择两个不同维度
dim_1 = "TenureGroup"
dim_2 = "Complain"
# TODO 2：使用groupby + agg完成双维分析
cross_analysis = df.groupby([dim_1, dim_2]).agg(
    用户数=("CustomerID", "nunique"),
    流失人数=("Churn", lambda x: (x == 1).sum()),
    流失率=("Churn", lambda x: (x == 1).mean()),
    平均订单数=("OrderCount", "mean")  # 额外行为指标
).reset_index()
# TODO 3：新增“样本提示”列
# 用户数<30标记为“小样本”，否则标记为“可观察”
def sample_tag(row):
    if row["用户数"] < 30:
        return "小样本"
    else:
        return "可观察"
cross_analysis["样本提示"] = cross_analysis.apply(sample_tag, axis=1)
# TODO 4：按流失率或用户数排序并展示
cross_analysis = cross_analysis.sort_values(by="流失率", ascending=False)
display(cross_analysis)

,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,样本提示
1,0-6个月,1,659,375,0.57,2.65,可观察
0,0-6个月,0,1491,322,0.22,2.43,可观察
7,7-12个月,1,406,81,0.20,2.67,可观察
3,13-24个月,1,414,52,0.13,3.35,可观察
6,7-12个月,0,1178,75,0.06,2.78,可观察
2,13-24个月,0,1053,43,0.04,3.85,可观察
4,24个月以上,0,304,0,0.00,3.75,可观察
5,24个月以上,1,125,0,0.00,3.06,可观察


In [16]:
# 检查点4：双维度交叉分析

assert dim_1 in allowed_cross_fields and dim_2 in allowed_cross_fields, \
    "两个分析维度必须来自允许字段"
assert dim_1 != dim_2, "两个分析维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), \
    "cross_analysis应为DataFrame"

required_cross_cols = {
    dim_1,
    dim_2,
    "用户数",
    "流失率",
    "样本提示",
}

assert required_cross_cols.issubset(cross_analysis.columns), \
    f"双维分析表缺少字段：{required_cross_cols - set(cross_analysis.columns)}"
assert cross_analysis["用户数"].sum() == len(df), \
    "双维组合用户数之和应等于总用户数"
assert set(cross_analysis["样本提示"]).issubset(
    {"小样本", "可观察"}
), "样本提示只能是“小样本”或“可观察”"

expected_sample_hint = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)
assert np.array_equal(
    cross_analysis["样本提示"].to_numpy(),
    expected_sample_hint,
), "样本提示与用户数阈值不一致"

print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的维度组合：**
> TODO：TenureGroup（生命周期）+ Complain（是否投诉）
>
**该组合的用户数、流失率和比较对象：**
> TODO：目标组合为 0-6 个月新用户且有投诉（Complain=1），总用户数 659 人，原始流失率 0.57（即 57%）。
横向对比同生命周期无投诉人群：0-6 个月、Complain=0，用户 1491 人，流失率仅 0.22（22%），二者流失率相差35%；
纵向对比其余有投诉分组：7-12 个月投诉用户流失率 0.20、13-24 个月投诉用户流失率 0.13、24 个月以上投诉用户流失率 0，0-6 个月投诉用户的流失风险显著高于全生命周期其他同类人群。
**是否存在小样本风险：**
>
> TODO:不存在小样本风险；判断依据：该分组用户数 659，远大于 30，样本提示标记为 “可观察”，样本量充足，数据趋势具备参考意义。
**为什么不能直接写成因果结论：**
>
> TODO:仅能观察到[新用户 + 投诉]与高流失高度相关，无法证明投诉是流失的直接原因。存在混淆变量干扰，例如新用户本身对平台容错度低，较差的下单体验既引发投诉，也促使用户流失，所以只能描述关联特征，不能断定 “投诉直接导致流失”。

## 任务5：输出统计报表（必做）

In [17]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

已输出： output\day05_analysis\overall_metrics.csv
已输出： output\day05_analysis\segment_analysis.csv
已输出： output\day05_analysis\cross_analysis.csv


In [18]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

通过：overall_metrics.csv，形状为(10, 2)
通过：segment_analysis.csv，形状为(4, 7)
通过：cross_analysis.csv，形状为(8, 7)
检查点5通过


## 任务6：结论、限制与建议（必做）

### 结论1

在____用户中，____指标为____，与____相比____。对应证据表：____。

> TODO：在0-6 个月新用户中，有投诉行为的用户流失率为0.57（57%），与同生命周期无投诉行为的用户相比高出 35 个百分点，流失风险显著更高。

### 结论2

> TODO：在全量平台用户中，总体流失率为0.1684（16.84%），用户平均订单数为 15.23 单，整体用户消费活跃度处于中等水平。

### 结论3

> TODO：在24 个月以上的成熟用户中，无论是否有投诉行为，用户流失率均为 0，与0-6 个月新用户群体相比流失风险完全消失，消费行为也稳定保持在较高水平。

### 分析限制

至少写明一项当前数据不能支持的分析，或一项可能影响结论的限制。

> TODO：当前数据为截面数据，仅能观察到生命周期、投诉行为与流失率的相关关系，无法排除订单频次、满意度、平台优惠力度等混淆变量的影响，不能证明投诉行为是新用户高流失的直接原因；同时数据未包含用户行为的时间序列变化，无法追踪用户流失的动态过程，难以判断用户投诉与流失的先后因果顺序。

### 运营建议与验证方式

提出一项与分析结果对应的建议，并说明还需要哪些数据或方法验证效果。

> TODO：运营建议：针对 0-6 个月有投诉行为的新用户，推出专属客诉安抚与首单补偿福利活动，优先响应该群体的客诉需求，降低该核心高流失风险群体的流失率。

>验证方式：将 0-6 个月有投诉行为的新用户随机分为实验组（推送客诉安抚 + 专属补偿福利）与对照组（无额外干预），持续追踪 30 天内两组用户的流失率、复购率、平均订单金额差异，验证运营干预的实际效果。

## 拓展任务（选做）

In [19]:
# 可选方向：
# 1. 使用qcut或业务规则构建订单活跃度分层；
# 2. 将双维分析整理为第6天绘图使用的长表；
# 3. 对一个反直觉结果提出两种数据核查方法；
# 4. 增加一项不与必做任务重复的业务分析。

# TODO（选做）

## 最终检查：GitHub提交前验收

In [20]:
required_files = [
    ROOT / "notebooks" / "day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

本地提交文件检查通过
请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。


，### GitHub提交清单

- [ ] 已填写姓名和专题；
- [ ] Notebook已重启内核并从头运行成功；
- [ ] 所有检查点均已通过；
- [ ] `output/day05_analysis/`中包含三个CSV；
- [ ] CSV中没有`Unnamed`索引列；
- [ ] 至少完成3条结论、1条限制和1项建议；
- [ ] 没有把返现写成消费额；
- [ ] 没有把相关关系写成确定因果关系；
- [ ] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析中最重要的数据发现是什么？
>0-6 个月有投诉行为的新用户共 659 人，流失率高达 57%，同生命周期无投诉新用户流失率仅 22%，二者流失差距显著；而 24 个月以上全量成熟用户无论是否投诉，流失率均为 0。

2. 哪个检查点最能帮助你发现错误？
>检查点2，用户流失分析的核心基准就是根据全局总体流失率，如果错了后续所有分层对比分析都会失去参考标准，因此这条检查最能帮助发现错误。

3. 哪条结论最容易被误解为因果关系？

>0-6 个月有投诉用户流失率远高于无投诉用户这条结论，极易被误读为 “用户投诉直接造成用户流失”，实际仅为相关关系，无法排除体验差、平台服务缺陷等混淆变量的共同影响。

4. 如果增加一个字段，你最希望增加什么？
>用户投诉发生时间字段；可精准判断投诉发生在流失前还是流失后，区分先后时序，辅助区分相关与因果，解决当前截面数据无法判断事件顺序的短板。

5. 第6天准备把哪张统计表转化为图表？为什么？
>cross_analysis.csv生命周期 + 投诉双维度交叉统计表；

> 表格多分组数据对比不直观，转化为分组柱状图可直观展示不同生命周期下，投诉 / 无投诉两类用户的流失率差异，清晰凸显新用户投诉群体的高流失风险，可视化后更便于运营快速识别高风险人群。
